# 01 Single Logic Run Dashboard

Use one signal source and one fixed execution configuration to inspect cluster behavior before optimization.

In [ ]:
from pathlib import Path
import sys

def find_project_root():
    candidates = [Path.cwd().resolve(), Path('Z:/SEN05_Autotrading'), Path('//10.11.12.6/Share/SEN05_Autotrading')]
    for candidate in candidates:
        current = candidate
        while True:
            if (current / 'pyproject.toml').exists() and (current / 'backtest_optimize').exists():
                return current
            if current.parent == current:
                break
            current = current.parent
    raise RuntimeError('Could not find project root.')

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BACKTEST_ROOT = project_root / 'backtest_optimize'
RAW_SIGNALS = project_root / 'raw_signals'
OUTPUT_DIR = BACKTEST_ROOT / 'outputs' / 'single_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
from IPython.display import HTML, Markdown, display

from backtest_optimize.contracts import AmbiguityPolicy, MarketSpec
from backtest_optimize.io.signal_loader import load_signal_csv
from backtest_optimize.io.market_data import load_ohlcv_from_core
from backtest_optimize.execution.engine import run_configured_backtest
from backtest_optimize.analysis.chart_payload import build_signal_chart_payload, render_signal_chart_html
from backtest_optimize.analysis.metrics import clusters_to_frame, summarize
from backtest_optimize.analysis.notebook_dashboard import (
    build_execution_config, cluster_review_frame, cluster_review_html,
    component_catalog_frame, control_panel_frame, dashboard_cards_html,
    dashboard_interpretation_frame, dashboard_metric_frame,
    discover_signal_catalog, run_context_frame, select_signal,
    status_breakdown_frame, style_dashboard_frame, style_report,
    warning_notes_frame,
)
from backtest_optimize.analysis.versioning import make_run_id, save_snapshot

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

In [ ]:
# Signal source controls
SIGNAL_STRATEGY = 'combo'
SIGNAL_CHOICE = 'auto'
PREFERRED_SYMBOL = 'US30'
PREFERRED_TIMEFRAME = 'H4'
WARMUP_BARS = 0

# Execution component controls
ENGINE = 'component'
PROFILE_NAME = 'combo_ctrader_v0'
ENTRY_MODEL = 'stop_breakout_signal_bar'
ENTRY_PARAMS = {'x_offset': 10.0}
SL_METHOD = 'signal_bar_atr_buffer'
SL_PARAMS = {'ksl_level': 2}
TP_METHOD = 'fib_atr_ctrader_v0'
TP_PARAMS = {'ktp_level': 6, 'exit_mode': 'fixed_only'}
ORDER_MODEL = 'stop_order'
ORDER_PARAMS = {'cancel_after_bars': 3}
EXIT_MODEL = 'fixed_only'
EXIT_PARAMS = {'sma_period': 20}

# Risk and market controls
ACCOUNT_SIZE = 10_000.0
RISK_PER_CLUSTER = 0.01
AMBIGUITY_POLICY = AmbiguityPolicy.CONSERVATIVE
MARKET_SPEC = MarketSpec(
    symbol=PREFERRED_SYMBOL, pip_size=1.0, pip_value_per_lot=1.0,
    min_lot=0.01, lot_step=0.01,
    commission_per_lot_per_side=0.0,
    spread_buffer_pips=0.0, slippage_buffer_pips=0.0,
)

signal_catalog = discover_signal_catalog(RAW_SIGNALS, SIGNAL_STRATEGY)
selected_signal = select_signal(
    signal_catalog, choice=SIGNAL_CHOICE,
    symbol=PREFERRED_SYMBOL, timeframe=PREFERRED_TIMEFRAME,
)
SIGNAL_FILE = selected_signal.path
SYMBOL = selected_signal.symbol
TIMEFRAME = selected_signal.timeframe
MARKET_SPEC = MarketSpec(**{**MARKET_SPEC.__dict__, 'symbol': SYMBOL})

RUN_CONFIG = build_execution_config(
    engine=ENGINE, profile_name=PROFILE_NAME,
    account_size=ACCOUNT_SIZE, risk_per_cluster=RISK_PER_CLUSTER,
    ambiguity_policy=AMBIGUITY_POLICY,
    entry_model=ENTRY_MODEL, entry_params=ENTRY_PARAMS,
    sl_method=SL_METHOD, sl_params=SL_PARAMS,
    tp_method=TP_METHOD, tp_params=TP_PARAMS,
    order_model=ORDER_MODEL, order_params=ORDER_PARAMS,
    exit_model=EXIT_MODEL, exit_params=EXIT_PARAMS,
)

display(Markdown('### Available Components'))
display(style_report(component_catalog_frame()))
display(Markdown('### Signal Catalog'))
display(style_report(signal_catalog.drop(columns=['path'], errors='ignore')))
display(Markdown('### Selected Controls'))
display(style_report(control_panel_frame(
    selection=selected_signal, run_config=RUN_CONFIG,
    market_spec=MARKET_SPEC, warmup_bars=WARMUP_BARS,
)))

In [ ]:
signals = load_signal_csv(SIGNAL_FILE, symbol=SYMBOL, timeframe=TIMEFRAME)
start = signals['bartime'].min()
end = signals['bartime'].max() + pd.Timedelta(days=10)
bars = load_ohlcv_from_core(
    SYMBOL, TIMEFRAME, start=start, end=end,
    warmup_bars=WARMUP_BARS, tail_bars=5,
)

display(Markdown('### Data Context'))
display(style_report(run_context_frame(
    signals=signals, bars=bars, selection=selected_signal,
    run_config=RUN_CONFIG,
)))
display(signals.head())
display(bars.head())

In [ ]:
chart_payload = build_signal_chart_payload(
    bars=bars, signals=signals, symbol=SYMBOL,
    timeframe=TIMEFRAME, max_bars=3000,
)
display(HTML(render_signal_chart_html(chart_payload, height=760)))

In [ ]:
result = run_configured_backtest(
    signals=signals, bars=bars, symbol=SYMBOL, timeframe=TIMEFRAME,
    market_spec=MARKET_SPEC, run_config=RUN_CONFIG,
)
summary = summarize(result)
clusters = clusters_to_frame(result)

display(Markdown('## Backtest Dashboard'))
display(HTML(dashboard_cards_html(summary, RUN_CONFIG, MARKET_SPEC)))
display(style_dashboard_frame(dashboard_interpretation_frame(summary, RUN_CONFIG, MARKET_SPEC)))
display(style_dashboard_frame(dashboard_metric_frame(summary)))
display(style_dashboard_frame(status_breakdown_frame(clusters)))
display(style_dashboard_frame(warning_notes_frame(summary, RUN_CONFIG, MARKET_SPEC)))

CLUSTER_REVIEW_MODE = 'first'
CLUSTER_REVIEW_LIMIT = 80
review = cluster_review_frame(clusters, limit=CLUSTER_REVIEW_LIMIT, mode=CLUSTER_REVIEW_MODE)
display(HTML(cluster_review_html(review)))

In [ ]:
RUN_ID = make_run_id('single', symbol=SYMBOL, timeframe=TIMEFRAME)
clusters_path = OUTPUT_DIR / (RUN_ID + '_clusters.csv')
summary_path = OUTPUT_DIR / (RUN_ID + '_summary.csv')
clusters.to_csv(clusters_path, index=False)
pd.DataFrame([summary]).to_csv(summary_path, index=False)

snapshot_path = save_snapshot(
    name=RUN_ID, run_id=RUN_ID, run_type='single',
    config={
        'strategy': SIGNAL_STRATEGY, 'symbol': SYMBOL, 'timeframe': TIMEFRAME,
        'run_config': RUN_CONFIG, 'market_spec': MARKET_SPEC,
    },
    result_summary=summary,
    outputs={'clusters': clusters_path, 'summary': summary_path},
    signal_file=SIGNAL_FILE,
    market_data_source_id='core_python.data.loader',
    assumptions=result.assumptions, repo_root=project_root,
)
print('run_id:', RUN_ID)
print(clusters_path)
print(summary_path)
print(snapshot_path)